# GROW Point-Sample Demo

Demonstrates all six no-auth `env-data-mcp` sources against five real GROW stream-water
sampling locations.  Each tool is called with the sample's location and a ±7-day window
around the collection date.

| Source | Tool | Latency |
|---|---|---|
| NASA POWER | `nasa_power_query` | ~1–5 s |
| SSURGO | `ssurgo_query` | ~2–5 s |
| SoilGrids v2 | `soilgrids_query` | ~2–5 s |
| GBIF occurrences | `gbif_occurrences` | ~5–30 s |
| OpenAQ | `openaq_query` | ~2–10 s (auth required) |
| Sentinel-5P TROPOMI | `sentinel5p_query` | ~30–120 s per date |

> **Network required.** Runs are skipped gracefully when a source is unreachable.

In [ ]:
import json

from env_data_mcp.sources.gbif import gbif_occurrences
from env_data_mcp.sources.nasa_power import nasa_power_query
from env_data_mcp.sources.openaq import openaq_query
from env_data_mcp.sources.sentinel5p import sentinel5p_query
from env_data_mcp.sources.soilgrids import soilgrids_query
from env_data_mcp.sources.ssurgo import ssurgo_query

# Five GROW samples used throughout (matches tests/conftest.py GROW_SAMPLES).
GROW_SAMPLES = [
    {"name": "Yukon_2004-3", "date": "2004-06-15", "lat": 61.93333333, "lon": -162.8666667},
    {"name": "Yukon_2004-1", "date": "2004-04-07", "lat": 61.93333333, "lon": -162.8666667},
    {"name": "yakimariver_2019", "date": "2019-08-19", "lat": 46.2531882, "lon": -119.4768203},
    {"name": "whiteclaycreek2_2019", "date": "2019-08-12", "lat": 39.8594333, "lon": -75.7839486},
    {"name": "whiteclaycreek1_2019", "date": "2019-08-12", "lat": 39.8577967, "lon": -75.7830393},
]


def date_window(date: str, days: int = 7) -> tuple[str, str]:
    """Return (start, end) dates as ±days around the given date."""
    from datetime import datetime, timedelta

    d = datetime.strptime(date, "%Y-%m-%d")
    start = (d - timedelta(days=days)).strftime("%Y-%m-%d")
    end = (d + timedelta(days=days)).strftime("%Y-%m-%d")
    return start, end


def show(label: str, result: dict) -> None:
    m = result["_meta"]
    status = "✓" if m["success"] else "✗"
    rows = m.get("rows_returned", len(result.get("data", [])))
    lat_s = m["latency_s"]
    err = f" — {m['error']}" if m.get("error") else ""
    print(f"  {status} {label:35s} rows={rows:4d}  latency={lat_s:.2f}s{err}")


print("Imports OK.")

## 1. NASA POWER — daily weather (all five samples)

In [ ]:
nasa_results = {}
for s in GROW_SAMPLES:
    start, end = date_window(s["date"])
    r = nasa_power_query(latitude=s["lat"], longitude=s["lon"], start_date=start, end_date=end)
    nasa_results[s["name"]] = r
    show(s["name"], r)

# Peek at one result.
sample_r = nasa_results["yakimariver_2019"]
first = sample_r["data"][0] if sample_r["data"] else {}
print(f"\nSample record (yakima): {json.dumps(first, indent=2)}")

## 2. SSURGO — soil map unit (US samples only; others return empty gracefully)

In [ ]:
for s in GROW_SAMPLES:
    r = ssurgo_query(latitude=s["lat"], longitude=s["lon"])
    show(s["name"], r)

## 3. SoilGrids v2 — global soil properties (all five samples)

In [ ]:
for s in GROW_SAMPLES:
    r = soilgrids_query(latitude=s["lat"], longitude=s["lon"])
    show(s["name"], r)

## 4. GBIF occurrences — biodiversity (all five samples, 50 km radius)

In [ ]:
for s in GROW_SAMPLES:
    start, end = date_window(s["date"], days=180)  # wider window — occurrences are sparse
    r = gbif_occurrences(
        latitude=s["lat"],
        longitude=s["lon"],
        radius_km=50.0,
        start_date=start,
        end_date=end,
        limit=50,
    )
    show(s["name"], r)
    if r["data"]:
        print(f"    species example: {r['data'][0].get('species', 'N/A')}")
    print(f"    license(s): {r['_meta']['license']}")

## 5. OpenAQ — surface air quality

OpenAQ v3 requires a free API key (`OPENAQ_API_KEY` env var).  
If the key is absent, the tool returns a structured `auth_present=False` response rather than raising.

In [ ]:
for s in GROW_SAMPLES:
    start, end = date_window(s["date"])
    r = openaq_query(
        latitude=s["lat"],
        longitude=s["lon"],
        radius_km=100.0,
        start_date=start,
        end_date=end,
        limit=100,
    )
    auth_note = " (no API key)" if not r["_meta"].get("auth_present", True) else ""
    show(s["name"] + auth_note, r)

## 6. Sentinel-5P TROPOMI — atmospheric columns

Each granule is ~150 MB; this cell queries **one date for one sample** (yakima, CO product)  
to keep total runtime within the notebook timeout.  
For a full 5-sample run, call `sentinel5p_query` per sample in a loop.

In [ ]:
yakima = GROW_SAMPLES[2]  # yakimariver_2019
r = sentinel5p_query(
    latitude=yakima["lat"],
    longitude=yakima["lon"],
    start_date=yakima["date"],
    end_date=yakima["date"],
    product="CO",
)
show(yakima["name"], r)
if r["data"]:
    rec = r["data"][0]
    co_key = next((k for k in rec if k.startswith("CO")), None)
    print(f"  CO total column: {rec.get(co_key, 'N/A')} {rec.get('CO_units', '')}")
    print(f"  granule: {rec.get('granule_id', 'N/A')}")
else:
    print("  No swath coverage on this date — try an adjacent date.")

## Summary

Each `_meta` block contains the licence, query parameters, and latency —  
sufficient provenance to reproduce any result or log it to a Lakehouse.

In [ ]:
yakima_nasa = nasa_results["yakimariver_2019"]
print("Meta for yakima NASA POWER query:")
print(json.dumps({k: v for k, v in yakima_nasa["_meta"].items() if k != "variable_info"}, indent=2))